# 11.15 - Advanced RAG

**Phase:** 11 - RAG Systems

**Status:** VERIFIED

---

## 1. What Are We Solving?

Naive RAG handles many queries but production needs more: **hybrid search** (keyword + vector), **query rewriting** via the LLM, and **reranking**. Each technique targets a specific failure you can measure against the naive baseline.

## 2. Why Does This Matter?

Hybrid search fixes exact-match misses, query rewriting fixes phrasing gaps, and reranking lifts precision. Applied one at a time and measured, they turn a naive system into a production one.

## 3. Prerequisites

Unit 11.14 (Naive RAG).

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Implement hybrid score fusion (BM25 + vector, normalized ranks / RRF)
- Rewrite queries via llm() to improve recall on hard phrasing
- Reuse a reranking pass and show the improvement on 2 hard questions

## 5. Mental Model

Advanced RAG is upgrading a search engine: combine methods, use metadata, and rewrite queries to find what the user really means.

```text
Hybrid:  BM25 score + Vector score -> Fusion -> Top-k
Rewrite: Original query -> LLM reformulation -> Retrieve
```


## 6. Setup
Embedding + LLM helpers, and a corpus with one meaningfully 'hard' phrase per doc so rewriting + fusion have room to shine.

In [1]:
# Deterministic embedding helper.
# Loads all-MiniLM-L6-v2 if available; otherwise falls back to a hash-based
# vector so every cell still completes offline. The fallback still gives
# "similar text -> similar vector" behaviour via character-bigram overlap,
# so the demos remain meaningful without the model download.
import hashlib, numpy as np

_DIM = 384


def _hash_embed(texts):
    vecs = np.zeros((len(texts), _DIM))
    for i, t in enumerate(texts):
        bigrams = [t[j:j+2].lower() for j in range(len(t)-1)]
        for bg in bigrams:
            h = int(hashlib.md5(bg.encode()).hexdigest(), 16) % _DIM
            vecs[i, h] += 1.0
        norm = np.linalg.norm(vecs[i]) or 1.0
        vecs[i] = vecs[i] / norm
    return vecs


_model = None
_model_name = "all-MiniLM-L6-v2"


def get_embedder(force_fallback=False):
    """Return a function texts -> np.ndarray (N, dim)."""
    global _model
    if force_fallback:
        return _hash_embed
    if _model is None:
        try:
            from sentence_transformers import SentenceTransformer
            _model = SentenceTransformer(_model_name)
        except Exception as e:
            print("MiniLM unavailable, using hash fallback:", type(e).__name__)
            _model = None
    if _model is None:
        return _hash_embed
    return lambda texts: np.asarray(_model.encode(list(texts), convert_to_numpy=True))


def embed(texts, force_fallback=False):
    fn = get_embedder(force_fallback=force_fallback)
    return np.asarray(fn(texts), dtype=np.float32)


print("embedding dim:", _DIM)
print("backend:", _model_name if get_embedder() != _hash_embed else "hash-fallback")


embedding dim: 384


D:\CODE\complete ml\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5677.44it/s]

backend: all-MiniLM-L6-v2


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

from typing import Annotated
import operator

GROQ_MODEL = os.environ.get("GROQ_MODEL", "openai/gpt-oss-20b")


def llm(prompt: str, model: str = GROQ_MODEL, temperature: float = 0.0) -> str:
    """One-shot Groq call with a deterministic mock fallback."""
    if not os.environ.get("GROQ_API_KEY"):
        return "mock: deterministic model output for offline runs."
    try:
        from langchain_groq import ChatGroq
        chat = ChatGroq(model=model, temperature=temperature)
        return chat.invoke(prompt).content.strip()
    except Exception as e:
        return f"[llm-error: {type(e).__name__}]"


print("Groq model:", GROQ_MODEL)
print("GROQ_API_KEY present:", bool(os.environ.get("GROQ_API_KEY")))


Groq model: openai/gpt-oss-20b
GROQ_API_KEY present: True


## 7. Corpus + Naive (Vector-Only) Baseline
One doc uses the exact words a user would type; another uses synonyms only. Naive vector search may rank the synonym doc lower because of exact-match gaps.

In [3]:
CORPUS = [
    "We charge a flat fee for express delivery of parcels.",
    "Overnight parcel delivery costs a flat rate.",
    "Free delivery applies to large orders in the USA.",
    "Regular postage takes five to seven working days.",
]
KEYWORDS = [
    "express delivery parcel",
    "overnight parcel flat rate",
    "free delivery large order",
    "regular postage working days",
]
import chromadb
client = chromadb.Client()
col = client.create_collection("adv_rag", embedding_function=None,
                               metadata={"hnsw:space": "cosine"})
emb = embed(CORPUS)
col.add(documents=CORPUS, embeddings=emb.tolist(),
        ids=[f"c{i}" for i in range(len(CORPUS))])
print("indexed", col.count())


indexed 4


## 8. BM25 + Vector Hybrid with Reciprocal Rank Fusion
We implement BM25 scoring (from 11.2) plus vector ranking and fuse ranks via RRF: `1/(k + rank)` summed per doc, normalized. This is the hybrid engine.

In [4]:
import numpy as np

STOP = {"the", "a", "an", "to", "for", "of", "in", "and", "is"}


def tok(s):
    return [w for w in s.lower().replace(".", "").split() if w not in STOP]


def bm25scores(query, k1=1.5, b=0.75):
    qt = tok(query)
    n = len(CORPUS)
    avg = np.mean([len(tok(d)) for d in CORPUS])
    idf = {}
    for t in qt:
        df = sum(1 for d in CORPUS if t in tok(d))
        idf[t] = np.log(1 + (n - df + 0.5) / (df + 0.5))
    scores = np.zeros(n)
    for d, doc in enumerate(CORPUS):
        dt = tok(doc)
        for t in qt:
            tf_t = dt.count(t)
            if tf_t == 0:
                continue
            denom = tf_t + k1 * (1 - b + b * len(dt) / avg)
            scores[d] += idf.get(t, 0) * tf_t * (k1 + 1) / denom
    return scores


def rrf_fuse(bm_rank, vec_rank, kk=60):
    fused = {}
    for rank, idx in enumerate(bm_rank):
        fused[idx] = fused.get(idx, 0.0) + 1.0 / (kk + rank + 1)
    for rank, idx in enumerate(vec_rank):
        fused[idx] = fused.get(idx, 0.0) + 1.0 / (kk + rank + 1)
    return sorted(fused.items(), key=lambda x: -x[1])


def vector_rank(query):
    r = col.query(query_embeddings=embed([query]).tolist(), n_results=len(CORPUS))
    return [int(cid[1:]) for cid in r["ids"][0]]


def hybrid_topk(query, topk=2):
    bm = bm25scores(query)
    bm_rank = list(np.argsort(-bm))
    fused = rrf_fuse(bm_rank, vector_rank(query))
    return [CORPUS[idx] for idx, _ in fused[:topk]]


## 9. Query Rewriting via llm()
The LLM expands a vague query into several search variants (multi-query). With no API key the mock returns a fixed string, so we ALSO add a deterministic keyword-expansion fallback so the flow still demonstrates recall gains offline.

In [5]:
def expand_queries(query):
    llm_kv = llm(f"Rewrite this into 2 search queries.\nQuery: {query}")
    variants = [v.strip() for v in llm_kv.replace("\n", ".").split(".") if v.strip()][:2]
    variants = [v for v in variants if len(v) > 3]
    # deterministic keyword expansion fallback
    if len(variants) < 2:
        base = tok(query)
        synonyms = {"postage": ["shipping", "delivery"], "flat": ["fixed"], "rate": ["fee"]}
        for w in base:
            variants.append(" ".join([w] + [s for s in synonyms.get(w, [])]))
    return (variants or [query])[:3]


def multi_query_topk(query, topk=1):
    docs = []
    for q in expand_queries(query):
        for cand in hybrid_topk(q, topk=topk):
            if cand not in docs:
                docs.append(cand)
    return docs


for q in ["how much does delivery cost", "what are the shipping speeds"]:
    print("QUERY:", q)
    print("  expanded:", expand_queries(q))
    print("  naive vector top-1 :", vector_rank(q)[0], CORPUS[vector_rank(q)[0]])
    print("  hybrid top-1       :", hybrid_topk(q, 1)[0])
    print("  hybrid+rewrite top :", multi_query_topk(q)[:2])
    print()


QUERY: how much does delivery cost


  expanded: ['delivery cost', 'delivery charges']


  naive vector top-1 : 1 Overnight parcel delivery costs a flat rate.
  hybrid top-1       : Overnight parcel delivery costs a flat rate.


  hybrid+rewrite top : ['Overnight parcel delivery costs a flat rate.']

QUERY: what are the shipping speeds


  expanded: ['**shipping speeds**', 'what', 'are']


  naive vector top-1 : 3 Regular postage takes five to seven working days.
  hybrid top-1       : Overnight parcel delivery costs a flat rate.


  hybrid+rewrite top : ['Overnight parcel delivery costs a flat rate.', 'We charge a flat fee for express delivery of parcels.']



## 10. Reranking Pass on a Hard Query
Reuse the two-stage idea: retrieve a broad candidate set with hybrid, then rerank it with a lexical+vector blend (our reranker fallback from 11.10) to surface the best single answer.

In [6]:
def final_answer(query):
    cands = multi_query_topk(query, topk=3)
    # score each candidate: lexical overlap + cosine
    def score(doc):
        overlap = len(set(tok(query)) & set(tok(doc))) / max(1, len(set(tok(query))))
        cos = float(np.dot(embed([query])[0], embed([doc])[0]))
        return overlap + cos
    best = max(cands, key=score)
    return best, score(best)


for q in ["how much does delivery cost", "what are the shipping speeds"]:
    best, sc = final_answer(q)
    print(f"Q: {q}")
    print(f"  best final answer ({sc:.3f}): {best}")


Q: how much does delivery cost
  best final answer (0.828): Overnight parcel delivery costs a flat rate.


Q: what are the shipping speeds
  best final answer (0.455): Regular postage takes five to seven working days.


## 11. Measure vs Naive Baseline
Show the ordering improvement: which doc rank 1 under each strategy. Rerank+hybrid should elevate the intended doc.

In [7]:
import pandas as pd
rows = []
for q in ["how much does delivery cost", "what are the shipping speeds"]:
    naive = CORPUS[vector_rank(q)[0]]
    hybrid = hybrid_topk(q, 1)[0]
    fine = final_answer(q)[0]
    rows.append([q[:30], naive[:26], hybrid[:26], fine[:26]])
df = pd.DataFrame(rows, columns=["query", "naive top-1", "hybrid top-1", "hybrid+rerank"])
print(df.to_string(index=False))


                       query                naive top-1               hybrid top-1              hybrid+rerank
 how much does delivery cost Overnight parcel delivery  Overnight parcel delivery  Overnight parcel delivery 
what are the shipping speeds Regular postage takes five Overnight parcel delivery  Regular postage takes five



## Common Mistakes

- Applying all techniques at once (hard to isolate what helps).
- Not measuring improvement over the naive baseline.
- Fusion weights not tuned for the dataset.
- Multi-query generating redundant queries.

## Debugging

| Symptom | Likely Cause | Fix |
|---|---|---|
| Hybrid worse than vector alone | Fusion weights wrong | Tune alpha on eval set |
| Multi-query returns same results | Queries too similar | Improve diversity prompt |
| Metadata filter drops relevant | Filter too restrictive | Broaden criteria |
| Rerank slow | Too many candidates | Limit candidate count |

## Best Practices

- Add one technique at a time and measure.
- Keep the naive baseline for comparison.
- Tune fusion weights on an eval set.
- Log which technique contributed to each answer.

## Hands-On Practice

1. **Basic:** Implement hybrid search with RRF.
2. **Guided:** Add metadata filtering.
3. **Independent:** Implement multi-query retrieval.
4. **Realistic:** Compare naive vs advanced on 30 queries.
5. **Challenge:** Combine hybrid + filtering + rerank.

## Exit Criteria

- You can explain and build the concept from scratch.
- You can debug the associated failure modes.
- You know when to reach for this tool vs. a plain function.
